# IMPORTS

In [ ]:
import pickle as pkl
from torch import nn
import os
import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from matplotlib import cm
from GranPreprocessing import load_protein_graph_data, prepare_graph_data_for_training
from GranModel import DualOutputGRAN


# PARAMS

In [ ]:
# PARAMS
max_length = 50

parent_folder = "nanos_networkx_small"  # Update this to your data path - this is relative!
chunk_length = 50
max_proteins = 3000  # Limit number of proteins for faster execution


# More aggressive subsequence parameters
min_subseq_length = 20  # Even smaller minimum subsequence length
step_size = 1  # Much smaller step size for more overlap
subgraph_limit = max_proteins * 300 # max number of subgraphs


model_path = "dual_output_gran_model_residual.pt"  # Path for saving/loading model


# Set device
def get_device():
    """
    Returns the best available device for PyTorch operations.
    Priority: CUDA GPU > MPS (Apple Silicon) > CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")




# Model loading if it already exists

In [ ]:
def load_model(model_path, model, device):
    """
    Load a saved model - handles both formats

    Args:
        model_path: Path to the saved model
        model: Model instance to load the weights into
        device: Device to load the model to

    Returns:
        The loaded model
    """
    if not os.path.exists(model_path):
        print(f"No checkpoint found at {model_path}")
        return model

    try:
        # Try to load as a dictionary first
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            # Assume it's just the state dict
            model.load_state_dict(checkpoint)
        print(f"Successfully loaded model from {model_path}")
    except Exception as e:
        print(f"Error loading model: {e}")

    return model

# Load data -
some of this is just sanity checks and global things needed later, adjacency matrices, AA sequences

In [ ]:

# Load protein graph data directly
try:
    full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
        parent_folder, max_proteins, chunk_length
    )
except Exception as e:
    print(f"Error loading graph data: {e}")
    print("Current directory contains:", os.listdir())



# First graph debug
print("First few nodes of first graph:")
first_graph = full_graphs[0]
for i, node in enumerate(sorted(first_graph.nodes())[:5]):
    print(f"Node {node} attributes: {first_graph.nodes[node]}")

# Define a standard set of amino acids (all 20 standard ones)
STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
               'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

# Get unique amino acids from sequences but ensure we have at least the standard 20
UNIQUE_AA = set()
for seq in full_sequences:
    UNIQUE_AA.update(seq)

# Merge with standard AAs
UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
print(f"Unique amino acids: {len(UNIQUE_AA)}")
print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

# Prepare data for training using graph data
aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
    subgraphs, subsequences, UNIQUE_AA
)

print(f"Prepared data: {len(aa_sequences)} sequences, {len(adjacency_matrices)} adjacency matrices")


# Generation
this is actually not a protein generation but a protein subsequence generation

In [ ]:
def generate_protein(model, adjacency_matrix, node_features, device, max_length=max_length, unique_aa=None):
    """
    Generate both a protein sequence and adjacency matrix using the dual output GRAN model

    Args:
        model: Trained DualOutputGRAN model
        adjacency_matrix: Input adjacency matrix
        node_features: Input node features
        device: Device to run inference on
        max_length: Maximum sequence length to generate
        unique_aa: List of amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Prepare inputs
    if len(adjacency_matrix.shape) == 2:
        adjacency_matrix = adjacency_matrix.unsqueeze(0)
    if len(node_features.shape) == 2:
        node_features = node_features.unsqueeze(0)

    adjacency_matrix = adjacency_matrix.to(device)
    node_features = node_features.to(device)

    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(node_features, adjacency_matrix, max_length=max_length)

    # Decode sequence
    generated_ids = outputs['generated_sequence'][0]

    if unique_aa is None:
        amino_acids = "ACDEFGHIKLMNPQRSTVWYX"
    else:
        amino_acids = unique_aa

    protein_sequence = ""
    for aa_id in generated_ids:
        if aa_id.item() < len(amino_acids):
            protein_sequence += amino_acids[aa_id.item()]

    # Get predicted adjacency matrix
    predicted_adjacency = outputs['adjacency_matrix'][0].cpu().numpy()

    return {
        'protein_sequence': protein_sequence,
        'adjacency_matrix': predicted_adjacency
    }


In [ ]:
def generate_full_protein_from_subsequences(model, full_protein_length, subsequence_length,
                                            adjacency_matrix_template, node_features_template,
                                            device, UNIQUE_AA, overlap_strategy='average'):
    """
    Generate a full protein by combining multiple subsequences with proper three-letter code handling
    """
    print(f"\nGenerating full protein of length {full_protein_length}...")

    # Create an empty adjacency matrix for the full protein
    full_adjacency = np.zeros((full_protein_length, full_protein_length))
    full_sequence = [''] * full_protein_length

    # Track how many times each position has been filled (for averaging)
    if overlap_strategy == 'average':
        adjacency_counts = np.zeros((full_protein_length, full_protein_length))
        sequence_candidates = [[] for _ in range(full_protein_length)]

    # Step size for subsequences (controls overlap)
    step_size = subsequence_length // 2  # 50% overlap

    # List of valid amino acid three-letter codes
    valid_aa_codes = [
        'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
        'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X'
    ]

    # Generate subsequences to cover the full protein
    for start_pos in range(0, full_protein_length, step_size):
        end_pos = min(start_pos + subsequence_length, full_protein_length)
        actual_subsequence_length = end_pos - start_pos

        # Prepare input for this subsequence
        subsequence_adj = adjacency_matrix_template[:actual_subsequence_length, :actual_subsequence_length]
        subsequence_features = node_features_template[:actual_subsequence_length]

        # Generate subsequence
        results = generate_protein(
            model, subsequence_adj, subsequence_features, device,
            max_length=actual_subsequence_length,
            unique_aa=UNIQUE_AA
        )

        # Parse the generated sequence into proper three-letter codes
        generated_raw_seq = results['protein_sequence']
        generated_adj = results['adjacency_matrix']

        # Use regular expressions to extract valid amino acid codes
        import re
        # Find all valid three-letter codes in the generated sequence
        generated_seq = []
        pos = 0
        while pos < len(generated_raw_seq):
            matched = False
            for code in valid_aa_codes:
                if generated_raw_seq[pos:].startswith(code):
                    generated_seq.append(code)
                    pos += len(code)
                    matched = True
                    break
            if not matched:
                # If no valid code found, skip a character
                pos += 1

        # Ensure we have the correct number of amino acids
        if len(generated_seq) < actual_subsequence_length:
            # Fill remaining positions with X
            generated_seq.extend(['X'] * (actual_subsequence_length - len(generated_seq)))
        elif len(generated_seq) > actual_subsequence_length:
            # Trim excess
            generated_seq = generated_seq[:actual_subsequence_length]

        # Handle sequence placement
        if overlap_strategy == 'average':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length:
                    sequence_candidates[start_pos + i].append(aa)
        elif overlap_strategy == 'use_first':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length and full_sequence[start_pos + i] == '':
                    full_sequence[start_pos + i] = aa
        elif overlap_strategy == 'use_last':
            for i, aa in enumerate(generated_seq):
                if start_pos + i < full_protein_length:
                    full_sequence[start_pos + i] = aa

        # Handle adjacency matrix placement
        for i in range(len(generated_seq)):
            for j in range(len(generated_seq)):
                if start_pos + i < full_protein_length and start_pos + j < full_protein_length:
                    if i < generated_adj.shape[0] and j < generated_adj.shape[1]:
                        if overlap_strategy == 'average':
                            full_adjacency[start_pos + i, start_pos + j] += generated_adj[i, j]
                            adjacency_counts[start_pos + i, start_pos + j] += 1
                        else:
                            full_adjacency[start_pos + i, start_pos + j] = generated_adj[i, j]

    # Post-process the results
    if overlap_strategy == 'average':
        # Average overlapping regions
        adjacency_counts[adjacency_counts == 0] = 1  # Avoid division by zero
        full_adjacency /= adjacency_counts

        # Choose most frequent amino acid for each position
        for i, candidates in enumerate(sequence_candidates):
            if candidates:
                # Count occurrences and pick most frequent
                from collections import Counter
                full_sequence[i] = Counter(candidates).most_common(1)[0][0]

    # Fill any remaining empty positions
    for i in range(full_protein_length):
        if full_sequence[i] == '':
            full_sequence[i] = 'X'

    # Join sequence with spaces for readability
    formatted_sequence = ' '.join(full_sequence)

    print(f"Generated sequence has {len(full_sequence)} amino acids")

    return {
        'full_sequence': formatted_sequence,
        'full_adjacency_matrix': full_adjacency
    }

In [ ]:

def generate_full_protein_sequence_and_structure(model, full_graphs, full_sequences, subgraphs,
                                                 subsequences, adjacency_matrices, node_features,
                                                 device, UNIQUE_AA):
    """Generate a full protein and visualize results"""
    if len(subgraphs) > 0:
        # Use the first subsequence to get chunk size
        reference_protein_length = len(subsequences[0])

        # Get the length of the first FULL protein, not subsequence
        if len(full_sequences) > 0:
            target_full_length = len(full_sequences[0])  # This is the actual full protein length
        else:
            # Fallback if full sequences not available
            target_full_length = 150  # Default value

        print(f"Target full protein length: {target_full_length}")
        print(f"Subsequence length: {reference_protein_length}")

        # Generate a full protein
        full_protein_results = generate_full_protein_from_subsequences(
            model=model,
            full_protein_length=target_full_length,
            subsequence_length=reference_protein_length,
            adjacency_matrix_template=adjacency_matrices[0],
            node_features_template=node_features[0],
            device=device,
            UNIQUE_AA=UNIQUE_AA,
            overlap_strategy='average'
        )

        print("\nGenerated full protein sequence:", full_protein_results['full_sequence'])
        print("Full adjacency matrix shape:", full_protein_results['full_adjacency_matrix'].shape)

        return full_protein_results



In [ ]:
def save_raw_pdb_from_coords(coords, sequence, filename="raw_protein.pdb"):
    """
    Convert 3D coordinates directly to PDB format without altering the structure

    Args:
        coords: Numpy array of shape (n_residues, 3) with 3D coordinates
        sequence: List or string of amino acid sequence
        filename: Output filename

    Returns:
        Path to the saved PDB file
    """
    # Ensure sequence is in one-letter code format
    if isinstance(sequence, list) and len(sequence) > 0 and len(sequence[0]) == 3:
        # Convert from three-letter to one-letter
        three_to_one = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y',
            'UNK': 'X', 'X': 'X'
        }
        one_letter_seq = [three_to_one.get(aa, 'X') for aa in sequence]
    elif isinstance(sequence, str):
        one_letter_seq = list(sequence)
    else:
        one_letter_seq = sequence

    # Convert one-letter to three-letter code
    one_to_three = {
        'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU', 'F': 'PHE',
        'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'K': 'LYS', 'L': 'LEU',
        'M': 'MET', 'N': 'ASN', 'P': 'PRO', 'Q': 'GLN', 'R': 'ARG',
        'S': 'SER', 'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
        'X': 'UNK'
    }

    with open(filename, 'w') as f:
        f.write("TITLE     Raw Coordinates Protein Structure\n")
        f.write("REMARK    Generated with preserved 3D coordinates\n")

        atom_index = 1
        ca_indices = []  # Track CA indices for CONECT records

        # Write CA atoms with exact coordinates
        for i, (coord, aa) in enumerate(zip(coords, one_letter_seq)):
            three_letter = one_to_three.get(aa, 'UNK')

            # Write CA atom with original coordinates
            f.write(f"ATOM  {atom_index:5d}  CA  {three_letter} A{i+1:4d}    "
                   f"{coord[0]:8.3f}{coord[1]:8.3f}{coord[2]:8.3f}  1.00  0.00           C  \n")
            ca_indices.append(atom_index)
            atom_index += 1

        # Add CONECT records based on adjacency matrix
        # You would need to pass the adjacency matrix as an additional parameter
        # Or derive it from the coordinates

        # For now, just connect sequential residues
        for i in range(len(ca_indices) - 1):
            f.write(f"CONECT{ca_indices[i]:5d}{ca_indices[i+1]:5d}\n")

        f.write("END\n")

    print(f"Raw PDB file saved with preserved coordinates to {filename}")
    return filename

In [ ]:

# keep
def save_as_pdb(coords, sequence, filename="generated_protein.pdb"):
    """
    Convert 3D coordinates and sequence to PDB format with proper backbone connectivity

    Args:
        coords: Numpy array of shape (n_residues, 3) with 3D coordinates
        sequence: List or string of amino acid sequence
        filename: Output filename

    Returns:
        Path to the saved PDB file
    """
    # Ensure sequence is in the right format
    if isinstance(sequence, list) and len(sequence) > 0 and len(sequence[0]) == 3:
        # Convert three-letter codes to one-letter
        three_to_one = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y',
            'UNK': 'X', 'X': 'X'
        }
        one_letter_seq = [three_to_one.get(aa, 'X') for aa in sequence]
    elif isinstance(sequence, str):
        one_letter_seq = list(sequence)
    else:
        one_letter_seq = sequence

    # Convert one-letter to three-letter for PDB format
    one_to_three = {
        'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU', 'F': 'PHE',
        'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'K': 'LYS', 'L': 'LEU',
        'M': 'MET', 'N': 'ASN', 'P': 'PRO', 'Q': 'GLN', 'R': 'ARG',
        'S': 'SER', 'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
        'X': 'UNK'
    }

    # First, make sure consecutive CA-CA distances are reasonable
    improved_coords = []
    target_ca_ca_distance = 3.8  # Angstroms

    for i in range(len(coords)):
        if i == 0:
            improved_coords.append(coords[i])
        else:
            prev_ca = improved_coords[i-1]
            curr_ca = coords[i]
            vec = curr_ca - prev_ca
            dist = np.linalg.norm(vec)

            if dist < 3.5 or dist > 4.1:
                vec = vec / max(dist, 0.001)  # Avoid division by zero
                new_ca = prev_ca + vec * target_ca_ca_distance
                improved_coords.append(new_ca)
            else:
                improved_coords.append(curr_ca)

    improved_coords = np.array(improved_coords)

    with open(filename, 'w') as f:
        f.write("TITLE     Connected Protein Structure\n")
        f.write("REMARK    Generated with proper connectivity\n")

        atom_index = 1
        atom_indices = {}  # To store atom indices for connectivity

        # Generate all atoms
        for i, (ca_coord, aa) in enumerate(zip(improved_coords, one_letter_seq)):
            three_letter = one_to_three.get(aa, 'UNK')
            res_atoms = {}

            # Calculate local coordinate system
            if i > 0 and i < len(improved_coords) - 1:
                # Use neighboring CAs to establish a local frame
                prev_ca = improved_coords[i-1]
                this_ca = ca_coord
                next_ca = improved_coords[i+1]

                # Create a local coordinate system
                forward = next_ca - prev_ca
                forward = forward / np.linalg.norm(forward)

                # Roughly up from the CA
                up = np.cross(forward, np.array([1.0, 0.0, 0.0]))
                if np.linalg.norm(up) < 0.1:
                    up = np.cross(forward, np.array([0.0, 1.0, 0.0]))
                up = up / np.linalg.norm(up)

                # Perpendicular to both
                right = np.cross(up, forward)
                right = right / np.linalg.norm(right)
            else:
                # For terminal residues, use simpler placement
                if i > 0:  # Last residue
                    prev_ca = improved_coords[i-1]
                    this_ca = ca_coord
                    forward = this_ca - prev_ca
                else:  # First residue
                    if len(improved_coords) > 1:
                        this_ca = ca_coord
                        next_ca = improved_coords[i+1]
                        forward = next_ca - this_ca
                    else:
                        # Single residue case
                        forward = np.array([1.0, 0.0, 0.0])

                forward = forward / np.linalg.norm(forward)
                up = np.cross(forward, np.array([1.0, 0.0, 0.0]))
                if np.linalg.norm(up) < 0.1:
                    up = np.cross(forward, np.array([0.0, 1.0, 0.0]))
                up = up / np.linalg.norm(up)
                right = np.cross(up, forward)
                right = right / np.linalg.norm(right)

            # Calculate atom positions
            n_offset = -forward * 1.32 + up * 0.45
            n_coord = ca_coord + n_offset

            c_offset = forward * 1.52 + up * 0.15
            c_coord = ca_coord + c_offset

            o_offset = forward * 0.6 + up * 1.06
            o_coord = c_coord + o_offset

            # Write atoms to PDB
            # N atom
            f.write(f"ATOM  {atom_index:5d}  N   {three_letter} A{i+1:4d}    "
                   f"{n_coord[0]:8.3f}{n_coord[1]:8.3f}{n_coord[2]:8.3f}  1.00  0.00           N  \n")
            res_atoms['N'] = atom_index
            atom_index += 1

            # CA atom
            f.write(f"ATOM  {atom_index:5d}  CA  {three_letter} A{i+1:4d}    "
                   f"{ca_coord[0]:8.3f}{ca_coord[1]:8.3f}{ca_coord[2]:8.3f}  1.00  0.00           C  \n")
            res_atoms['CA'] = atom_index
            atom_index += 1

            # C atom
            f.write(f"ATOM  {atom_index:5d}  C   {three_letter} A{i+1:4d}    "
                   f"{c_coord[0]:8.3f}{c_coord[1]:8.3f}{c_coord[2]:8.3f}  1.00  0.00           C  \n")
            res_atoms['C'] = atom_index
            atom_index += 1

            # O atom
            f.write(f"ATOM  {atom_index:5d}  O   {three_letter} A{i+1:4d}    "
                   f"{o_coord[0]:8.3f}{o_coord[1]:8.3f}{o_coord[2]:8.3f}  1.00  0.00           O  \n")
            res_atoms['O'] = atom_index
            atom_index += 1

            # CB atom (except for glycine)
            if three_letter != "GLY":
                cb_offset = right * 1.1 + up * 0.7
                cb_coord = ca_coord + cb_offset

                f.write(f"ATOM  {atom_index:5d}  CB  {three_letter} A{i+1:4d}    "
                       f"{cb_coord[0]:8.3f}{cb_coord[1]:8.3f}{cb_coord[2]:8.3f}  1.00  0.00           C  \n")
                res_atoms['CB'] = atom_index
                atom_index += 1

            # Store atom indices for this residue
            atom_indices[i] = res_atoms

        # Write CONECT records for backbone connectivity
        for i in range(len(improved_coords)):
            current = atom_indices[i]

            # N-CA bond
            f.write(f"CONECT{current['N']:5d}{current['CA']:5d}\n")

            # CA-C bond
            f.write(f"CONECT{current['CA']:5d}{current['C']:5d}\n")

            # C-O bond
            f.write(f"CONECT{current['C']:5d}{current['O']:5d}\n")

            # CA-CB bond (except for glycine)
            if 'CB' in current:
                f.write(f"CONECT{current['CA']:5d}{current['CB']:5d}\n")

            # Peptide bond to next residue
            if i > 0:
                prev = atom_indices[i-1]
                f.write(f"CONECT{prev['C']:5d}{current['N']:5d}\n")

        # End of file
        f.write("END\n")

    print(f"Connected PDB file saved to {filename}")
    return filename




In [ ]:
# Create the dual output model

hidden_dim = 128  # Fixed dimension to ensure compatibility
num_layers = 2
n_heads = 4
amino_acid_vocab_size = len(UNIQUE_AA)

model = DualOutputGRAN(
        node_features=38,  # Now 38
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        n_heads=n_heads,
        dropout=0.1,
        amino_acid_vocab_size=amino_acid_vocab_size
    ).to(device)

model = load_model(model_path, model, device)

# Generate full protein (not just subsequence)
full_results = generate_full_protein_sequence_and_structure(
        model, full_graphs, full_sequences, subgraphs, subsequences,
        adjacency_matrices, node_features, device, UNIQUE_AA
    )




# Main Pipeline Run of model

In [ ]:

#keep
def improved_coords_from_adjacency(adjacency_matrix, sequence, max_iter=3000, lr=0.01,
                                   random_seed=42, sharpness=15.0):
    """
    Improved algorithm to generate 3D coordinates from an adjacency matrix
    with better initialization and balanced optimization for local and global contacts

    Args:
        adjacency_matrix: Binary adjacency matrix (N x N)
        sequence: Amino acid sequence (list or string)
        max_iter: Maximum optimization iterations
        lr: Learning rate
        random_seed: Random seed for initialization
        sharpness: Parameter for contact prediction sharpness

    Returns:
        3D coordinates for each residue (N x 3)
    """
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    N = adjacency_matrix.shape[0]
    print(f"Generating 3D coordinates for {N} residues...")

    # Convert adjacency to tensor
    adj_tensor = torch.tensor(adjacency_matrix, dtype=torch.float32)

    # Create repulsion mask (all non-diagonal elements)
    repulsion_mask = (torch.eye(N) == 0)

    # Initialize coordinates - try multiple initializations
    best_coords = None
    best_loss = float('inf')

    # Try 3 different initializations and pick the best one
    for init_attempt in range(3):
        print(f"Initialization attempt {init_attempt+1}...")

        if init_attempt == 0:
            # First try: random spherical initialization
            coords = torch.randn(N, 3)
            # Normalize to push points to a spherical shell
            norms = torch.norm(coords, dim=1, keepdim=True)
            coords = coords / norms * np.sqrt(N)  # Scale based on protein size
        elif init_attempt == 1:
            # Second try: spiral initialization (good for globular proteins)
            phi = torch.linspace(0, 10*np.pi, N)
            theta = torch.linspace(0, 4*np.pi, N)
            r = torch.linspace(1, 3, N)
            coords = torch.zeros((N, 3))
            coords[:, 0] = r * torch.sin(theta) * torch.cos(phi)
            coords[:, 1] = r * torch.sin(theta) * torch.sin(phi)
            coords[:, 2] = r * torch.cos(theta)
        else:
            # Third try: Linear + noise initialization
            coords = torch.zeros((N, 3))
            for i in range(N):
                coords[i, 0] = i * 3.8  # Start with CA-CA distance
            # Add random noise
            coords = coords + torch.randn(N, 3) * 2.0

        coords.requires_grad = True

        # Create optimizer with slightly higher learning rate for better exploration
        optimizer = torch.optim.Adam([coords], lr=lr*2)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=50, cooldown=50, min_lr=1e-5
        )

        # Optimization loop
        prev_loss = float('inf')
        patience_counter = 0

        for step in range(max_iter):
            optimizer.zero_grad()

            # Calculate pairwise distances
            dists = torch.cdist(coords, coords, p=2)

            # Contact satisfaction loss - higher weight for long-range contacts
            contact_loss = 0
            local_weight = 1.0
            medium_weight = 2.0
            long_weight = 3.0  # Emphasize long-range contacts more!

            # Create masks for different contact ranges
            seq_dists = torch.abs(torch.arange(N).unsqueeze(1) - torch.arange(N).unsqueeze(0))
            local_mask = (seq_dists <= 5) & (seq_dists > 0)
            medium_mask = (seq_dists > 5) & (seq_dists <= 20)
            long_mask = (seq_dists > 20)

            # Calculate losses for each range
            if local_mask.sum() > 0:
                local_contacts = adj_tensor[local_mask]
                local_distances = dists[local_mask]
                logits = sharpness * (8.0 - local_distances)
                local_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, local_contacts
                )
                contact_loss += local_weight * local_loss

            if medium_mask.sum() > 0:
                medium_contacts = adj_tensor[medium_mask]
                medium_distances = dists[medium_mask]
                logits = sharpness * (8.0 - medium_distances)
                medium_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, medium_contacts
                )
                contact_loss += medium_weight * medium_loss

            if long_mask.sum() > 0:
                long_contacts = adj_tensor[long_mask]
                long_distances = dists[long_mask]
                logits = sharpness * (8.0 - long_distances)
                long_loss = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, long_contacts
                )
                contact_loss += long_weight * long_loss

            # Sequential connectivity loss - ensuring connected residues are close
            seq_dists = torch.diag(dists, diagonal=1)
            connectivity_loss = torch.mean((seq_dists - 3.8) ** 2)  # ~3.8Å is typical CA-CA distance

            # Repulsion loss to prevent collapse
            min_dist = 3.6  # Minimum distance between any two CA atoms
            min_dist_violation = torch.relu(min_dist - dists)
            repulsion_loss = (min_dist_violation[repulsion_mask] ** 2).sum()

            # Radius of gyration restraint (to keep it compact)
            center_of_mass = torch.mean(coords, dim=0)
            dists_from_center = torch.norm(coords - center_of_mass, dim=1)
            max_radius = np.sqrt(N)  # Heuristic for protein radius
            radius_loss = torch.mean(torch.relu(dists_from_center - max_radius) ** 2)

            # Add volume exclusion term for more compact packing
            volume_loss = torch.mean(torch.exp(-dists[repulsion_mask] / 2.0))

            # Combined loss
            total_loss = (
                contact_loss +
                0.1 * connectivity_loss +
                0.05 * repulsion_loss +
                0.01 * radius_loss +
                0.01 * volume_loss
            )

            # Backwards pass
            total_loss.backward()

            # Gradient clipping to prevent instability
            torch.nn.utils.clip_grad_norm_([coords], max_norm=1.0)

            optimizer.step()
            scheduler.step(total_loss)

            # Print progress
            if step % 200 == 0:
                print(f"Step {step:5d} | Loss: {total_loss.item():.4f} | Contact: {contact_loss.item():.4f}")

                # Calculate contact satisfaction rate for monitoring
                with torch.no_grad():
                    contact_points = (adj_tensor > 0.5).float()
                    proximity = (dists < 8.0).float()
                    correct_contacts = (contact_points * proximity).sum()
                    total_contacts = contact_points.sum()
                    if total_contacts > 0:
                        satisfaction_rate = 100 * correct_contacts / total_contacts
                        print(f"Contact satisfaction: {satisfaction_rate.item():.2f}%")

            # Early stopping check
            if abs(prev_loss - total_loss.item()) < 1e-5:
                patience_counter += 1
                if patience_counter > 50:
                    print(f"Early stopping at step {step}")
                    break
            else:
                patience_counter = 0

            prev_loss = total_loss.item()

        # Check if this attempt gave better results
        with torch.no_grad():
            # Calculate final contact satisfaction
            dists = torch.cdist(coords, coords, p=2)
            contact_points = (adj_tensor > 0.5).float()
            proximity = (dists < 8.0).float()
            correct_contacts = (contact_points * proximity).sum()
            total_contacts = contact_points.sum()
            if total_contacts > 0:
                satisfaction_rate = correct_contacts / total_contacts
                print(f"Final contact satisfaction: {satisfaction_rate.item()*100:.2f}%")

                # Calculate final loss
                final_loss = total_loss.item()
                print(f"Final loss: {final_loss:.4f}")

                # Check if this is the best attempt so far
                if final_loss < best_loss:
                    best_loss = final_loss
                    best_coords = coords.detach().clone()
                    print(f"New best coordinates found with loss {best_loss:.4f}")

    # Return the best coordinates
    final_coords = best_coords.detach().numpy()
    print(f"Coordinate generation complete with final loss: {best_loss:.4f}")

    return final_coords

def apply_pymol_visualization_script(pdb_path, script_path="protein_visualization.pml"):
    """
    Create a PyMOL script to visualize the protein structure

    Args:
        pdb_path: Path to the PDB file
        script_path: Path to save the PyMOL script

    Returns:
        Path to the script file
    """
    with open(script_path, 'w') as f:
        f.write(f"""# PyMOL script for protein visualization
load {pdb_path}, protein
hide everything
show cartoon
color skyblue, protein
show sticks, protein
set stick_radius, 0.3
set cartoon_fancy_helices, 1
set cartoon_highlight_color, grey50
set stick_color, blue
set sphere_scale, 0.25
show spheres, name CA
color purple, name CA
set ray_opaque_background, off
set ray_shadows, 0
set spec_reflect, 0.25
set antialias, 2
bg_color white
center protein
zoom protein
# Create nice view
set_view (\\
    1.000000000,    0.000000000,    0.000000000,\\
    0.000000000,    1.000000000,    0.000000000,\\
    0.000000000,    0.000000000,    1.000000000,\\
    0.000000000,    0.000000000,  -50.000000000,\\
    0.000000000,    0.000000000,    0.000000000,\\
    40.000000000,  100.000000000,  -20.000000000 )
# Display help in PyMOL
cmd.wizard("message", "Use A to toggle between all representation styles")
cmd.set_key("A", "cmd.hide('everything', 'protein');cmd.show('cartoon', 'protein')")
cmd.set_key("S", "cmd.hide('everything', 'protein');cmd.show('sticks', 'protein')")
cmd.set_key("D", "cmd.hide('everything', 'protein');cmd.show('spheres', 'protein')")
cmd.set_key("F", "cmd.hide('everything', 'protein');cmd.show('surface', 'protein')")
cmd.set_key("G", "cmd.hide('everything', 'protein');cmd.show('dots', 'protein')")
""")

    print(f"PyMOL script saved to {script_path}")
    print(f"Run with: pymol {script_path}")
    return script_path

In [ ]:
# keep
def generate_pdb_from_model_output_fixed(results, output_dir="."):
    """
    Generate PDB file from model output with improved coordinate generation

    Args:
        results: Dictionary with 'full_sequence' and 'full_adjacency_matrix'
        output_dir: Directory to save output files

    Returns:
        Dictionary with paths to saved files
    """


    os.makedirs(output_dir, exist_ok=True)

    generated_seq = results['full_sequence']
    generated_adj = results['full_adjacency_matrix']

    # Generate 3D coordinates with improved algorithm
    print("Generating 3D coordinates with improved algorithm...")

    # Binary adjacency for coordinate generation
    binary_adjacency = (generated_adj > 0.065).astype(np.float32)

    # Generate coordinates using the improved algorithm
    coords = improved_coords_from_adjacency(
        adjacency_matrix=binary_adjacency,
        sequence=generated_seq,
        max_iter=3000,
        lr=0.01
    )

    # Save outputs
    output_files = {}

    # 1. Save as standard PDB with improved coordinates
    pdb_path = os.path.join(output_dir, "generated_protein.pdb")
    output_files['pdb'] = save_as_pdb(coords, generated_seq, pdb_path)

    # 2. Save as full-atom PDB with improved coordinates
    full_pdb_path = os.path.join(output_dir, "generated_protein_full.pdb")
    #output_files['full_pdb'] = save_full_atom_pdb(coords, generated_seq, full_pdb_path)

    # 3. Save raw coordinates as simple PDB
    raw_pdb_path = os.path.join(output_dir, "raw_protein.pdb")
    output_files['raw_pdb'] = save_raw_pdb_from_coords(coords, generated_seq, raw_pdb_path)

    # 4. Save with adjacency matrix contacts
    adj_pdb_path = os.path.join(output_dir, "protein_with_contacts.pdb")
    #output_files['adj_pdb'] = save_pdb_with_adj_matrix(coords, generated_seq, binary_adjacency, adj_pdb_path)

    # 5. Save as XYZ (simpler format)
    xyz_path = os.path.join(output_dir, "generated_protein.xyz")
    #output_files['xyz'] = save_coords_as_xyz(coords, generated_seq, xyz_path)

    # 6. Save sequence as FASTA
    fasta_path = os.path.join(output_dir, "generated_protein.fasta")
    with open(fasta_path, 'w') as f:
        f.write(">generated_protein\n")
        f.write(generated_seq + "\n")
    output_files['fasta'] = fasta_path

    # 7. Create a PyMOL script for visualization
    pymol_script = os.path.join(output_dir, "visualize_protein.pml")
    apply_pymol_visualization_script(adj_pdb_path, pymol_script)
    output_files['pymol_script'] = pymol_script

    print(f"All outputs saved to {output_dir}")
    return output_files



In [ ]:
output_files = generate_pdb_from_model_output_fixed(
    results=full_results,
    output_dir='./protein_output_fixed'
)